# 02 · Baselines: order-free SVM and DTW template matching
Owner: **Member 1**

Two non-neural reference points that bracket the sequential question:

| Baseline | Uses temporal order? | Learns? | Why include it |
|---|---|---|---|
| **MFCC summary statistics + SVM** | No: means/stds over time | Yes | Tells us how far we get when the sequence is collapsed into a bag of frames. If the neural sequential models beat it, modelling order helps. |
| **DTW nearest-template (k-NN)** | Yes: aligns whole sequences | No | Classic isolated-word recogniser (Sakoe & Chiba, 1978); handles speaking-rate variation by warping time, but learns no invariance to speaker or noise. |

Model selection uses the **validation** split only (log loss, the official metric). The test split
is used once, at the end.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive to the (much faster) local Colab disk. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv`, `SampleSubmission.csv` and `Audio.zip`
(downloaded from Zindi) in a Google Drive folder called `swahili_audio`, and set `REPO_URL`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-audio-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-audio-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_audio")
    LOCAL_DATA = Path("/content/data")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if not (LOCAL_DATA / ".ready").exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        for f in DRIVE_DATA.glob("*.csv"):
            shutil.copy(f, LOCAL_DATA)
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_DATA / "Audio.zip"), "-d", str(LOCAL_DATA)], check=True)
        (LOCAL_DATA / ".ready").touch()
    os.environ["DATA_DIR"] = str(LOCAL_DATA)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")          # features persist between sessions
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")  # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src import config as C
from src.data import get_splits, label_encoder, encode, load_unlabelled, add_paths
from src.dtw import dtw_class_scores, scores_to_proba
from src.evaluate import compute_metrics, log_experiment, save_final_results, make_submission, show_log
from src.features import compute_features
from src.utils import set_seed, Timer

MEMBER = "Member 1"
set_seed(C.SEED)
df = get_splits()
classes, to_id = label_encoder(df.label)
K = len(classes)
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
ytr, yva, yte = (encode(d.label, to_id) for d in (tr, va, te))
zindi = add_paths(load_unlabelled())

## Part A · MFCC summary statistics + classifier (order-free)
Features per clip: mean, std, min, max of 20 MFCCs; mean and std of their deltas; mean and std of
spectral centroid, bandwidth, roll-off, zero-crossing rate and RMS; and duration.

In [ ]:
Xtr, Xva, Xte = (compute_features(d.path, "stats") for d in (tr, va, te))
print("feature matrix:", Xtr.shape)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=C.SEED)
svm_grid = {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.01]} if not C.QUICK else {"svc__C": [10]}
experiments_a = [
    dict(id="BL-01", description="MFCC stats + logistic regression",
         hypothesis="A linear model on order-free statistics gives a lower bound.",
         motivated_by="Starting point",
         make=lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))),
    dict(id="BL-02", description="MFCC stats + RBF-SVM (C, gamma tuned by 5-fold CV on train)",
         hypothesis="Non-linear decision boundaries separate words better than a linear model.",
         motivated_by="BL-01: check whether the gap is due to linearity",
         make=lambda: GridSearchCV(make_pipeline(StandardScaler(), SVC(probability=True, random_state=C.SEED)),
                                   svm_grid, cv=cv, scoring="neg_log_loss", n_jobs=-1)),
]

best_a = None
for e in experiments_a:
    with Timer() as t:
        clf = e["make"]().fit(Xtr, ytr)
    pv = clf.predict_proba(Xva)
    m = compute_metrics(yva, pv, K)
    cfg = getattr(clf, "best_params_", {})
    log_experiment(e["id"], MEMBER, "SVM baseline", e["description"], e["hypothesis"], cfg, m,
                   e["motivated_by"], t.seconds)
    print(f"{e['id']}: val log loss {m['log_loss']:.4f} | acc {m['accuracy']:.3f} | macro-F1 {m['macro_f1']:.3f} {cfg}")
    if best_a is None or m["log_loss"] < best_a[2]["log_loss"]:
        best_a = (e, clf, m)

### BL-03 · Does silence trimming matter?
EDA showed a large and variable share of silence. Re-run the best classifier on features computed
**without** trimming.

In [ ]:
Xtr_nt, Xva_nt = (compute_features(d.path, "stats", trim=False) for d in (tr, va))
with Timer() as t:
    clf_nt = experiments_a[1]["make"]().fit(Xtr_nt, ytr)
m_nt = compute_metrics(yva, clf_nt.predict_proba(Xva_nt), K)
log_experiment("BL-03", MEMBER, "SVM baseline", "MFCC stats + RBF-SVM, NO silence trimming",
               "Without trimming, duration/silence statistics are dominated by recording habits, not the word.",
               getattr(clf_nt, "best_params_", {}), m_nt, "EDA section 4: large, variable silence", t.seconds)
print(f"BL-03 (no trimming): val log loss {m_nt['log_loss']:.4f} vs {best_a[2]['log_loss']:.4f} with trimming")

## Part B · DTW template matching (sequence alignment, no learning)
Each query is aligned to a random set of training templates per class (MFCC + Δ + ΔΔ, per-utterance
mean/variance normalised, cosine distance, Sakoe-Chiba band). The class score is the mean cost of the
*k* nearest templates of that class; the softmax temperature that turns costs into probabilities is
tuned on validation, because log loss needs probabilities, not just a ranking.

In [ ]:
Str = compute_features(tr.path, "mfcc_var")
Sva = compute_features(va.path, "mfcc_var")
Ste = compute_features(te.path, "mfcc_var")
temps = np.geomspace(0.005, 0.5, 25)


def tune_temperature(S, y):
    losses = [compute_metrics(y, scores_to_proba(S, T), K)["log_loss"] for T in temps]
    return float(temps[int(np.argmin(losses))])


dtw_exps = [
    dict(id="DTW-01", description="DTW 1-NN, 10 templates/class", per_class=10, k=1,
         hypothesis="Alignment alone captures the word's temporal shape.", motivated_by="Classical baseline"),
    dict(id="DTW-02", description="DTW 5-NN, 30 templates/class", per_class=30, k=5,
         hypothesis="More templates cover more speakers; averaging k neighbours reduces sensitivity to one odd template.",
         motivated_by="DTW-01: a single template per decision is sensitive to speaker mismatch"),
]
if C.QUICK:
    for e in dtw_exps:
        e["per_class"] = min(e["per_class"], 5)

best_b = None
for e in dtw_exps:
    with Timer() as t:
        S = dtw_class_scores(Str, ytr, Sva, K, per_class=e["per_class"], k=e["k"], seed=C.SEED)
    T = tune_temperature(S, yva)
    m = compute_metrics(yva, scores_to_proba(S, T), K)
    log_experiment(e["id"], MEMBER, "DTW baseline", e["description"], e["hypothesis"],
                   {"per_class": e["per_class"], "k": e["k"], "temperature": T}, m, e["motivated_by"], t.seconds)
    print(f"{e['id']}: val log loss {m['log_loss']:.4f} | acc {m['accuracy']:.3f} | macro-F1 {m['macro_f1']:.3f}")
    if best_b is None or m["log_loss"] < best_b[1]["log_loss"]:
        best_b = (e, m, T)

In [ ]:
show_log("SVM baseline"), show_log("DTW baseline")

## Final evaluation on the held-out test split (once) + Zindi submissions

In [ ]:
e, clf, _ = best_a
pt = clf.predict_proba(Xte)
save_final_results("baseline_svm", "MFCC-stats + SVM", yte, pt, classes, te.file.values,
                   extra={"selected_experiment": e["id"]})
make_submission(zindi, clf.predict_proba(compute_features(zindi.path, "stats")), classes, "baseline_svm")

In [ ]:
e, _, T = best_b
S_te = dtw_class_scores(Str, ytr, Ste, K, per_class=e["per_class"], k=e["k"], seed=C.SEED)
save_final_results("baseline_dtw", "DTW k-NN", yte, scores_to_proba(S_te, T), classes, te.file.values,
                   extra={"selected_experiment": e["id"], "temperature": T})
S_z = dtw_class_scores(Str, ytr, compute_features(zindi.path, "mfcc_var"), K,
                       per_class=e["per_class"], k=e["k"], seed=C.SEED)
make_submission(zindi, scores_to_proba(S_z, T), classes, "baseline_dtw")

> **Write-up notes (edit after running):** Compare BL-01 vs BL-02 (linear vs non-linear),
> BL-02 vs BL-03 (effect of trimming) and SVM vs DTW (order-free vs order-aware but unlearned).
> Fill the `observation` column in `experiments/log_svm_baseline.csv` and `log_dtw_baseline.csv`.

In [ ]:
from src.utils import persist_outputs
persist_outputs()